# Custom CNN for Knee Osteoarthritis Classification

This notebook is a copy of our shared pipeline`00_shared_pipeline_metrics_test.ipynb`, version of 15 Oct, by Jobin. All cells are kept exactly as in the shared pipeline, except the ones marked **NEW** or **CHANGED**

**Before you start:** turn on the GPU: Runtime → Change runtime type → T4 GPU → Save

**Final training (CHANGED):** uses the best settings from our tuning (`01_custom_cnn_kaggle_round2.ipynb`, compared on the validation set):
- Class imbalance: **mild augmentation + weighted sampler** (as agreed in the team, same as `02_transfer_learning.ipynb`)
- Batch size **16**, learning rate **0.0001** (round 1), weight decay **0.00001**, dropout **0.5** (round 2)
- Up to **30 epochs** with early stopping (in tuning, the best epoch was often the last one)

**Quick test:** to check everything runs without training for long, set `EPOCHS = 1` in section 2 and skip section 7 (test set). Set it back to 30 for real training.

**!!Do not evaluate on the test set during development!!** Run section 7 only once, at the very end

## 1. Install packages and check the environment

Run the next two cells once per fresh Colab runtime. Use `uv` to install missing packages into this notebook's current Python environment, so imports work immediately

In [ ]:
!git clone https://github.com/michael-duer/medical-image-classification.git
%cd medical-image-classification


In [ ]:
from pathlib import Path
import sys

candidates = [
    Path("/content/medical-image-classification"),
    Path("/content/medical-image-classification/medical-image-classification")
]

PROJECT_ROOT = next(
    (p for p in candidates if (p / "data").is_dir()
     and (p / "src" / "metrics.py").is_file()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find dataset and metrics.py together.")

sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)
print("Dataset:", PROJECT_ROOT / "data")
print("Metrics module:", PROJECT_ROOT / "src" / "metrics.py")


In [ ]:
%pip -q install uv

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "PIL": "pillow",
    "tqdm": "tqdm",
}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run(["uv", "pip", "install", "--python", sys.executable, *missing], check=True)
else:
    print("All baseline dependencies are already installed.")
print("Notebook Python:", sys.executable)
print("If an import fails after installation, restart the runtime and rerun.")

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Settings

### Data
The example data was made on a Mac and contains hidden `._` files. They are deleted automatically, because the code would otherwise try to open them as images and crash.

### Overfitting
means the model memorizes the training images (including random details like noise or a label in the corner) instead of learning real signs of osteoarthritis. It then works well on training images but badly on new patients. Three new settings help against this:

- **`WEIGHT_DECAY = 1e-5`** (= 0.00001, best value of tuning round 2): pulls all weights toward 0 at every training step. Only patterns that help for many images keep large weights; random details of single images shrink away. The model stays simpler and generalizes better.
- **`DROPOUT = 0.5`**: during training, randomly switches off 50% of the features, so the model can't rely on single features. Automatically off during validation and testing.
- **`EARLY_STOP_PATIENCE = 7`**: stops training if the validation AUROC hasn't improved for 7 epochs.

### Early stopping
During training, the model usually gets better on the validation images at first. After some time it stops improving. If we keep training, it starts memorizing the training images, so the validation result gets worse again.

Early stopping prevents this:
1. After every epoch, we measure the validation AUROC
2. If it is the best so far, the model is saved.
3. If it has not improved for 7 epochs in a row, training stops automatically.
4. At the end, the best saved model is loaded not the last one.

**Why 7 and not 1?** The validation result jumps up and down a little from epoch to epoch. With a patience of 1, training would stop at the first small dip. 7 epochs give the model a fair chance to improve again.

`EPOCHS` is therefore only the maximum training often stops earlier.

### Other changes compared with the shared pipeline
- **`EPOCHS`**: was 1 (only a quick pipeline test), now 30 for real training. For a quick test use 1 or 2.
- **`LR = 1e-4`** (= 0.0001, was 0.001): best learning rate of tuning round 1. With 0.001 the validation AUROC was clearly lower (0.787 vs. 0.840 after 10 epochs).
- **`BATCH_SIZE = 16`**: best batch size of tuning round 1 (32 was not better).
- **`NUM_WORKERS`**: helpers that load images in the background. Use 0 on your own computer (VS Code), 2 in Colab (faster). If you get a "worker" error, set it to 0.

In [ ]:
DATA_DIR = Path('/content/medical-image-classification/data')
OUTPUT_DIR = Path('/content/medical-image-classification/output')

IMAGE_SIZE = 224
BATCH_SIZE = 16          # best batch size of tuning round 1
NUM_WORKERS = 2          # CHANGED (was 0): 2 helpers load images in the background -> faster. Set back to 0 if you get a "worker" error.
EPOCHS = 30             # CHANGED (was 1). For a quick test set EPOCHS = 1.
LR = 1e-4                # CHANGED (was 1e-3): best learning rate of tuning round 1
WEIGHT_DECAY = 1e-5      # NEW: small penalty against overfitting (best value of tuning round 2)
DROPOUT = 0.5            # NEW: randomly switch off 50% of features while training
EARLY_STOP_PATIENCE = 7  # NEW: stop if val AUROC does not improve for 7 epochs

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Set DATA_DIR to your actual dataset folder first: {DATA_DIR}")
print('Dataset path:', DATA_DIR.resolve())

### 2.1 Remove macOS metadata files (._*) from the dataset (DONT KNOW IF WE NEED THAT)

In [ ]:
from pathlib import Path

data_dir = Path("/content/medical-image-classification/data")

mac_files = list(data_dir.rglob("._*"))

print("Mac metadata files found:", len(mac_files))

for file in mac_files:
    file.unlink()

print("Removed Mac metadata files.")

## 3. Load and inspect data

The same label mapping and normalization will be used by every model. For fair comparison, save and reuse the same split manifest before individual experiments. If OUR data already comes with splits, this notebook respects them. If not, the fallback stratified image-level split below is for smoke-testing only and must be replaced when patients have multiple images.

The transform converts X-rays to 3-channel RGB so this data pipeline also works with standard ImageNet-pretrained CNNs later.

In [ ]:
from torchvision.datasets.folder import default_loader

# Normalization matches common ImageNet-pretrained ResNet/EfficientNet weights
# Keep these choices the same in all experiments
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

class ImageSubset(Dataset):
    """ImageFolder-compatible subset that stores explicit (path, label) entries."""
    def __init__(self, samples, transform):
        self.samples = [(str(p), int(y)) for p, y in samples]
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = default_loader(path)  # RGB
        return self.transform(img), label

existing_train = DATA_DIR / 'train'
existing_val = DATA_DIR / 'val'
existing_test = DATA_DIR / 'test'

if existing_train.is_dir() and existing_val.is_dir() and existing_test.is_dir():
    print('Using supplied train/val/test folders.')
    train_source = datasets.ImageFolder(existing_train)
    val_source = datasets.ImageFolder(existing_val)
    test_source = datasets.ImageFolder(existing_test)
    class_names = train_source.classes
    assert val_source.class_to_idx == train_source.class_to_idx == test_source.class_to_idx,         'Class folders do not match across splits.'
    split_samples = {
        'train': train_source.samples,
        'val': val_source.samples,
        'test': test_source.samples,
    }
else:
    print('No complete train/val/test layout found: creating illustrative image-level splits.')
    whole = datasets.ImageFolder(DATA_DIR)
    class_names = whole.classes
    paths = np.array([path for path, _ in whole.samples])
    labels = np.array([label for _, label in whole.samples])
    # 70% train / 15% val / 15% test, stratified by grade
    ix = np.arange(len(paths))
    train_ix, remainder_ix = train_test_split(
        ix, test_size=0.30, random_state=SEED, stratify=labels)
    val_ix, test_ix = train_test_split(
        remainder_ix, test_size=0.50, random_state=SEED,
        stratify=labels[remainder_ix])
    split_samples = {
        'train': [(paths[i], labels[i]) for i in train_ix],
        'val': [(paths[i], labels[i]) for i in val_ix],
        'test': [(paths[i], labels[i]) for i in test_ix],
    }
    print('WARNING: image-level splitting is inappropriate if patients have multiple images.')

num_classes = len(class_names)
assert num_classes >= 2, 'At least two grade folders are required.'
print('Class mapping:', {name: i for i, name in enumerate(class_names)})
for split, entries in split_samples.items():
    print(f'{split}: {len(entries)} images')

# For shared reproducibility: export the same manifest to all members
manifest = pd.DataFrame(
    [(split, str(path), int(label)) for split, entries in split_samples.items()
     for path, label in entries],
    columns=['split', 'path', 'label']
)
# Save to a shared persistent location AFTER confirming splits are appropriate
# manifest.to_csv('/content/drive/MyDrive/Knee_OA_Project/split_manifest.csv', index=False)


# CHANGED: class imbalance with mild augmentation + weighted sampler
# (team decision, same code as 02_transfer_learning.ipynb; replaces the rotation-balanced dataset)

# Mild augmentation for knee X-rays (training only)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(degrees=5),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.05,
        contrast=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# Validation and test transformations (no augmentation)
base_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

train_dataset = ImageSubset(
    split_samples["train"],
    train_transform
)

val_dataset = ImageSubset(
    split_samples["val"],
    base_transform
)

test_dataset = ImageSubset(
    split_samples["test"],
    base_transform
)

# Weighted sampler: rare grades are drawn more often (weight = 1 / number of images of that grade)
train_labels = np.array([
    label for _, label in split_samples["train"]
])
counts = np.bincount(train_labels, minlength=num_classes)
class_weights = 1.0 / counts
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(train_labels),
    replacement=True
)

print("Original class counts:", counts)
print("Class sampling weights:", class_weights)

loaders = {
    "train": DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,                      # sampler instead of shuffle
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    ),
    "val": DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    ),
    "test": DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda")
    ),
}

images, labels = next(iter(loaders["train"]))

print("Batch shape:", images.shape)

## 3.2Class Distribution Before and After Weighted Sampling

In [ ]:

from collections import Counter

original_labels = [
    label for path, label in split_samples["train"]
]

sampled_labels = []
for _, labels in loaders["train"]:
    sampled_labels.extend(labels.tolist())

print("Original distribution:")
print(Counter(original_labels))

print("\nSampled distribution (one epoch):")
print(Counter(sampled_labels))


## 3.3 Sample Image Visualization and Dataset Class Distribution

In [ ]:
# Show several unnormalized examples; no training required
fig, axes = plt.subplots(1, min(4, len(images)), figsize=(12, 3))
axes = np.atleast_1d(axes)
for ax, image, label in zip(axes, images, labels):
    display_image = image.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    ax.imshow(display_image.clip(0, 1))
    ax.set_title(class_names[int(label)])
    ax.axis('off')
plt.tight_layout()
plt.show()

print(pd.crosstab(manifest['split'], manifest['label']).rename(
    columns={i: name for i, name in enumerate(class_names)}))

### 3.4 Check the class distribution before and after sampling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Original training distribution
original_labels = [
    label for path, label in split_samples["train"]
]

original_counts = Counter(original_labels)

# Distribution after weighted sampling (one epoch)
sampled_labels = []

for _, labels in loaders["train"]:
    sampled_labels.extend(labels.numpy().tolist())

sampled_counts = Counter(sampled_labels)

# Display comparison
print("KL Grade | Original | After Sampling")
print("-------------------------------------")

for grade in range(5):
    print(
        f"KL{grade}      | "
        f"{original_counts[grade]:5d}    | "
        f"{sampled_counts[grade]:5d}"
    )

# Plot
grades = np.arange(5)
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    grades - width/2,
    [original_counts[i] for i in grades],
    width,
    label="Original"
)

plt.bar(
    grades + width/2,
    [sampled_counts[i] for i in grades],
    width,
    label="After Sampling"
)

plt.xticks(grades, [f"KL{i}" for i in grades])
plt.xlabel("Kellgren-Lawrence Grade")
plt.ylabel("Number of Images Sampled")
plt.title("Class Distribution Before and After Sampling")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Custom CNN (KneeCNN)

Replaces the `TinyCNN` test model of the shared pipeline. Like TinyCNN, the final layer outputs one logit (score) per grade; no softmax is used before the cross-entropy loss (the loss applies it internally).

### The two parts of the model
1. **`self.features`: What do I see?**
   5 `conv_block`s + average pooling → 256 numbers that summarize the X-ray
2. **`self.classifier`: Which grade is it?**
   Flatten → Dropout → Linear → 5 scores (KL0–KL4). The highest score is the prediction

`forward()` defines the order: first `features`, then `classifier`

### One `conv_block`, step by step
Each block takes `in_ch` feature maps and produces `out_ch` feature maps (= `out_ch` different patterns it looks for)

1. **Conv2d (3×3)**: A small 3×3 filter slides over the image, multiplies its 9 numbers with the 9 pixels below and adds them up. A high result means "pattern found here". The block learns `out_ch` different filters by itself during training (e.g. vertical edge, bright spot)
   - `padding=1` keeps the image size the same
   - `bias=False` because BatchNorm does that job
2. **BatchNorm2d** rescales the numbers of each feature map to a stable range → faster, more stable training (important when training from scratch)
3. **ReLU** negative values become 0, positive values stay. Keeps only "pattern found". Makes the network non-linear, so it can learn complex shapes
4. **Steps 1–3 again** the second convolution looks at the feature maps of the first one and combines patterns (e.g. edge + edge = corner). Two 3×3 convolutions see a 5×5 area with fewer parameters than one 5×5 filter (idea from VGG networks).
5. **MaxPool2d(2)** keeps the largest value of every 2×2 square → the image becomes half as wide and half as high. The strongest signals stay, computation gets faster, and the next block "sees" a larger part of the knee. Has no parameters.

### The whole model, step by step (typically finds that in each stetp)
1. **Input image:** 3 × 224×224
2. **Block 1:** 32 maps × 112×112 → ctypically finds edges, lines
3. **Block 2:** 64 maps × 56×56 →  corners, curves
4. **Block 3:** 128 maps × 28×28 → simple shapes
5. **Block 4:** 256 maps × 14×14 → parts of the knee (bone edge, joint gap)
6. **Block 5:** 256 maps × 7×7 → signs of osteoarthritis (spurs, narrowing)
7. **Average pooling:** 256 numbers summarizing the whole X-ray
8. **Dropout + Linear:** 5 scores → the highest score = predicted KL grade

The image gets smaller, but the number of feature maps grows → the network trades "where exactly" for "what is there"

### Differences to TinyCNN
- 5 instead of 3 blocks → sees larger structures like the whole joint space
- 2 convolutions per block→ richer patterns
- BatchNorm→ stable training
- Dropout (0.5) → less overfitting

**Size:** about 2.36 million learnable parameters (TinyCNN: ~23,000; ResNet-18: ~11 million). Most of them are in blocks 4 and 5 (256 channels). The cell below prints the exact number.


In [ ]:
def conv_block(in_ch, out_ch):
  #in_ch how many img come in, out_ch how many img come out, how many different patterns
    """Two 3x3 convolutions with BatchNorm + ReLU, then halve the image size."""
    return nn.Sequential(                                                #simply means do these layers one after the other
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),  # find patterns
        nn.BatchNorm2d(out_ch),                                          # stabilize numbers
        nn.ReLU(inplace=True),                                           # negatives -> 0
        nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),                                                 # halve the size
    )


class KneeCNN(nn.Module):
    """Custom CNN trained from scratch. Input: 3x224x224 image, output: one score per grade."""

    def __init__(self, n_classes, dropout=0.5):
        super().__init__()                                              #sets up PyTorch internals
        # Part 1: feature extractor - what is in the image?
        self.features = nn.Sequential(
            conv_block(3, 32),     # 224 -> 112 (edges, lines)
            conv_block(32, 64),    # 112 -> 56 (corners, curves)
            conv_block(64, 128),   # 56  -> 28 (simple shapes)
            conv_block(128, 256),  # 28  -> 14 (parts of knee)
            conv_block(256, 256),  # 14  -> 7 (signs of osteoarthriss)
            nn.AdaptiveAvgPool2d((1, 1)),   # each 7x7 map -> 1 number (average)
        )                                   #output numbers that summarize whole x-ray

        # Part 2: classifier - which KL grade?
        self.classifier = nn.Sequential(
            nn.Flatten(),                 #shape -> simple list
            nn.Dropout(dropout),          #dropout: during training randomly switch of 50% of the 256 numbers
            nn.Linear(256, n_classes),    #-> so mofel csn not rely on single feature
        )                                 #automatically off durinc validation/test

    def forward(self, x):
        return self.classifier(self.features(x))
        # forward() defines HOW data flows through the model
        # PyTorch calls it automatically when we write model(images)
        # First Part 1 (features), then Part 2 (classifier)

model = KneeCNN(num_classes, dropout=DROPOUT).to(DEVICE)
print('Learnable parameters:', sum(p.numel() for p in model.parameters()))

# Same check as in the shared pipeline- does the output have the right shape?
with torch.no_grad():
    output = model(images[:2].to(DEVICE))
print('Output shape:', tuple(output.shape))
assert output.shape == (min(2, len(images)), num_classes)

## 5. Training + validation loops (NEW training loop)

`run_epoch` and `get_predictions` are the **same as in the shared pipeline**.

Only the loop is new:

- **AdamW** optimizer = Adam (as before) + `WEIGHT_DECAY` against overfitting
- **Scheduler:** halves the learning rate if the validation AUROC does not improve for 3 epochs
- **Every epoch:** train, check on validation, compute validation **AUROC** (1.0 = perfect, 0.5 = guessing)
- **Best model** (highest validation AUROC) is saved; **early stopping** after `EARLY_STOP_PATIENCE` epochs without improvement.
- **At the end** the best model is loaded back

This takes the longest (5,778 training images per epoch, drawn by the weighted sampler; up to 30 epochs)

In [ ]:
# From the shared pipeline (unchanged)
def run_epoch(model, loader, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, count = 0.0, 0, 0
    for x, y in tqdm(loader, leave=False, desc='train' if is_training else 'validate'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(x)
            loss = criterion(logits, y)
            if is_training:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        count += y.size(0)
    return {'loss': total_loss / count, 'accuracy': correct / count}


@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    actual, probs = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        actual.append(y.numpy())
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(actual), np.concatenate(probs)


# NEW: full training with best-model saving and early stopping
CHECKPOINT_FILE = 'best_knee_cnn.pt'
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

history = []
best_auc = 0.0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_stats = run_epoch(model, loaders['train'], optimizer)   # learn
    val_stats = run_epoch(model, loaders['val'])                  # check

    y_val, val_prob = get_predictions(model, loaders['val'])
    val_auc = roc_auc_score(y_val, val_prob, multi_class='ovr', average='macro')
    scheduler.step(val_auc)

    history.append({'epoch': epoch + 1,
                    'train_loss': train_stats['loss'],
                    'train_accuracy': train_stats['accuracy'],
                    'val_loss': val_stats['loss'],
                    'val_accuracy': val_stats['accuracy'],
                    'val_auroc': val_auc})
    print(f"Epoch {epoch + 1:2d} | train acc {train_stats['accuracy']:.3f} "
          f"| val acc {val_stats['accuracy']:.3f} | val AUROC {val_auc:.3f}")

    if val_auc > best_auc:
        best_auc = val_auc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), CHECKPOINT_FILE)
        print('  -> new best model saved')
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print('Early stopping: validation AUROC stopped improving.')
            break

# Load the BEST model back (not the last one)
model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))
print(f'Best validation AUROC: {best_auc:.3f}')
pd.DataFrame(history)

## 6. Validation predictions and metrics (shared pipeline)

**!!!Do not evaluate the test split yet!!!**

For multiclass AUROC, use softmax probabilities and specify
**one-vs-rest, macro average**. If validation contains no samples for a class, AUROC may be undefined; the function reports that rather than inventing a score.



In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    actual, probs = [], []
    for x, y in loader:
        logits = model(x.to(DEVICE))
        actual.append(y.numpy())
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(actual), np.concatenate(probs)

def summarize_metrics(y_true, probabilities, class_names):
    predictions = probabilities.argmax(axis=1)
    results = {'accuracy': accuracy_score(y_true, predictions)}
    present = np.unique(y_true)
    if len(class_names) == 2 and len(present) == 2:
        results['auroc'] = roc_auc_score(y_true, probabilities[:, 1])
    elif len(present) == len(class_names):
        results['auroc'] = roc_auc_score(
            y_true, probabilities, labels=np.arange(len(class_names)),
            multi_class='ovr', average='macro')
    else:
        results['auroc'] = float('nan')
        print('AUROC undefined: not all classes appear in this validation split.')
    return results, confusion_matrix(y_true, predictions,
                                      labels=np.arange(len(class_names)))

y_val, val_prob = get_predictions(model, loaders['val'])
metrics, cm = summarize_metrics(y_val, val_prob, class_names)
print('Validation metrics:', metrics)
print('Confusion matrix (rows=true, cols=predicted):')
display(pd.DataFrame(cm, index=class_names, columns=class_names))

### 6.1 Import, Reload, and Verify the Metrics Module (shared pipeline)

In [ ]:
import sys
import importlib
import inspect

# Make sure Colab finds your project
sys.path.insert(0, "/content/medical-image-classification")

# Reload the updated metrics module
import src.metrics
importlib.reload(src.metrics)

# Import the updated function
from src.metrics import evaluate_predictions

# Check which file and function Colab is using
print("File:", inspect.getfile(evaluate_predictions))
print("Function:", inspect.signature(evaluate_predictions))

### 6.2 Verification of Metrics Module and Plotting Configuration (shared pipeline)

Makes sure the notebook uses the CURRENT version of src/metrics.py - only a check, nothing is calculated here

In [ ]:
import importlib
import inspect
import src.metrics

importlib.reload(src.metrics)

# Always use the function directly from the reloaded module
evaluate_predictions = src.metrics.evaluate_predictions

print("1. File:", inspect.getfile(evaluate_predictions))
print("2. Signature:", inspect.signature(evaluate_predictions))
print("3. Backend:", src.metrics.plt.get_backend())

source = inspect.getsource(evaluate_predictions)

print("4. Contains display(fig):", "display(fig)" in source)
print("5. Contains plt.show():", "plt.show()" in source)
print("6. Contains if show_plots:", "if show_plots:" in source)

## 7. Detailed CNN evaluation 8shared pipeline)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

plt.switch_backend("module://matplotlib_inline.backend_inline")

print("Current backend:", matplotlib.get_backend())

# Generate predictions
y_test, test_prob = get_predictions(model, loaders["test"])

print("True labels:", y_test.shape)
print("Predicted probabilities:", test_prob.shape)

import src.metrics

importlib.reload(src.metrics)
from src.metrics import evaluate_predictions

# Evaluate CNN performance
results = evaluate_predictions(
    y_true=y_test,
    y_prob=test_prob,
    model_name="KneeCNN",
    show_plots=True,
    save_plots=False
)

print(results)

## 8. Training and validation plots (shared pipeline)

In [ ]:
from src.metrics import plot_training_history

plot_training_history(
    history,
    output_dir=OUTPUT_DIR / "metrics",
    model_name="KneeCNN",
    show_plots = True,
    save_plots = False
)